# House Price Prediction in Google Colab

This notebook adapts the project for Google Colab. It checks the runtime, installs dependencies, loads data, trains the model, validates outputs, and optionally launches a lightweight Gradio demo.

## 1. Check Colab Runtime and Python Version

Use this section to confirm the active runtime, Python version, and whether a GPU is available.

In [ ]:
import os
import platform
import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)

try:
    gpu_info = subprocess.check_output(["bash", "-lc", "nvidia-smi --query-gpu=name --format=csv,noheader"], text=True).strip()
    print("GPU available:", gpu_info or "No GPU detected")
except Exception:
    print("GPU available: No GPU detected or nvidia-smi unavailable")

## 2. Install Required Packages in Colab

Install and pin the packages used by the notebook. Colab usually starts with a compatible Python version, but this step makes the environment reproducible.

In [ ]:
%pip install -q --upgrade pip
%pip install -q numpy==2.1.2 pandas==2.2.3 scikit-learn==1.5.2 joblib==1.4.2 gradio==5.18.0

import numpy as np
import pandas as pd
import sklearn
import joblib
import gradio as gr

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("gradio:", gr.__version__)

## 3. Load Project Files (Upload, Drive, or Git)

Choose one of the following options:
- Manual upload for quick tests.
- Google Drive for persistent files.
- GitHub clone for a clean project checkout.

In [ ]:
# Option A: manual upload for a custom housing.csv file.
# from google.colab import files
# uploaded = files.upload()

# Option B: mount Google Drive for persistent storage.
from google.colab import drive

try:
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped or already mounted:", exc)


/content/Houses-Price-Detection
Exists:

## 4. Configure Paths, Environment Variables, and Secrets

Set all file locations to Colab-safe paths and keep any secrets in environment variables or notebook inputs rather than hardcoding them.

In [ ]:
COLAB_PROJECT_ROOT = Path("/content/Houses-Price-Detection")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Houses-Price-Detection")

if PROJECT_DIR.exists():
    PROJECT_ROOT = PROJECT_DIR
elif DRIVE_PROJECT_ROOT.exists():
    PROJECT_ROOT = DRIVE_PROJECT_ROOT
else:
    PROJECT_ROOT = COLAB_PROJECT_ROOT

BACKEND_DIR = PROJECT_ROOT / "backend"
DATA_DIR = BACKEND_DIR / "data"
ARTIFACTS_DIR = BACKEND_DIR / "artifacts"
MODEL_PATH = ARTIFACTS_DIR / "house_price_model.joblib"
METRICS_PATH = ARTIFACTS_DIR / "metrics.json"
CUSTOM_DATA_PATH = DATA_DIR / "housing.csv"

os.environ.setdefault("HOUSE_PRICE_OUTPUT_DIR", str(ARTIFACTS_DIR))

# Secure pattern for secrets: set them in Colab's runtime variables or environment.
# Example:
# os.environ["MY_SECRET"] = "..."
SECRET_EXAMPLE = os.getenv("MY_SECRET", "")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Secret present:", bool(SECRET_EXAMPLE))

## 5. Adapt Local File I/O to Colab Storage

Use helper functions that read and write through `/content` or mounted Drive paths instead of Windows-only local paths.

In [ ]:
def ensure_directory(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_text(path: Path, content: str) -> None:
    ensure_directory(path.parent)
    path.write_text(content, encoding="utf-8")


def save_json(path: Path, payload: dict) -> None:
    import json

    ensure_directory(path.parent)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV file: {path}")
    return pd.read_csv(path)


def generate_demo_data(n_samples: int = 2000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    area_m2 = rng.uniform(35, 400, n_samples)
    bedrooms = rng.integers(1, 7, n_samples)
    bathrooms = rng.integers(1, 5, n_samples)
    floors = rng.integers(1, 4, n_samples)
    age_years = rng.integers(0, 80, n_samples)
    distance_to_center_km = rng.uniform(0.5, 35, n_samples)
    has_garage = rng.integers(0, 2, n_samples)
    has_garden = rng.integers(0, 2, n_samples)
    neighborhood_score = rng.uniform(1, 10, n_samples)

    noise = rng.normal(0, 15000, n_samples)
    price = (
        area_m2 * 2200
        + bedrooms * 15000
        + bathrooms * 12000
        + floors * 9000
        - age_years * 1100
        - distance_to_center_km * 3200
        + has_garage * 18000
        + has_garden * 12000
        + neighborhood_score * 25000
        + noise
    )

    price = np.maximum(price, 30000)

    return pd.DataFrame(
        {
            "area_m2": area_m2,
            "bedrooms": bedrooms,
            "bathrooms": bathrooms,
            "floors": floors,
            "age_years": age_years,
            "distance_to_center_km": distance_to_center_km,
            "has_garage": has_garage,
            "has_garden": has_garden,
            "neighborhood_score": neighborhood_score,
            "price": price,
        }
    )


def load_or_create_dataset() -> pd.DataFrame:
    ensure_directory(DATA_DIR)
    if CUSTOM_DATA_PATH.exists():
        print(f"Loading dataset from {CUSTOM_DATA_PATH}")
        return load_csv(CUSTOM_DATA_PATH)

    demo = generate_demo_data()
    save_text(DATA_DIR / "README.txt", "Drop a housing.csv file here to use your own data in Colab.")
    demo.to_csv(CUSTOM_DATA_PATH, index=False)
    print(f"Generated demo dataset at {CUSTOM_DATA_PATH}")
    return demo

print("I/O helpers ready.")

## 6. Run the Workflow End-to-End

This cell trains the model, evaluates it, and saves the main artifacts in the Colab session.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split


def train_house_price_model(df: pd.DataFrame):
    target_column = "price"
    feature_columns = [column for column in df.columns if column != target_column]

    X = df[feature_columns]
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=18,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    metrics = {
        "mae": float(mean_absolute_error(y_test, predictions)),
        "r2": float(r2_score(y_test, predictions)),
        "rows": int(len(df)),
    }

    ensure_directory(ARTIFACTS_DIR)
    joblib.dump(model, MODEL_PATH)
    save_json(METRICS_PATH, metrics)

    print("Training complete")
    print(metrics)
    print("Model saved to:", MODEL_PATH)
    return model, metrics, X_test, y_test


dataset = load_or_create_dataset()
model, metrics, X_test, y_test = train_house_price_model(dataset)
trained_feature_columns = [column for column in dataset.columns if column != "price"]

## 7. Validate with Quick Tests and Output Checks

Run lightweight checks to confirm the trained model, metrics, and saved files look correct in Colab.

In [ ]:
assert MODEL_PATH.exists(), f"Expected model artifact at {MODEL_PATH}"
assert METRICS_PATH.exists(), f"Expected metrics file at {METRICS_PATH}"
assert isinstance(metrics, dict) and "mae" in metrics and "r2" in metrics
assert X_test.shape[1] == len(trained_feature_columns)
assert len(trained_feature_columns) == 9

sample = X_test.iloc[[0]].copy()
prediction = model.predict(sample)[0]
assert float(prediction) > 0

print("Quick checks passed.")
print("Sample prediction:", float(prediction))
print("Metrics:", metrics)

## 8. Persist Results to Google Drive

Save the trained model, metrics, and dataset to Google Drive so they survive after the Colab session ends. This section also launches a small Gradio app for prediction.

In [ ]:
from shutil import copy2

DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "colab_outputs"
DRIVE_ARTIFACTS_DIR = DRIVE_OUTPUT_ROOT / "artifacts"
ensure_directory(DRIVE_ARTIFACTS_DIR)

for source_path in [MODEL_PATH, METRICS_PATH, CUSTOM_DATA_PATH]:
    if source_path.exists():
        copy2(source_path, DRIVE_ARTIFACTS_DIR / source_path.name)

print("Artifacts copied to:", DRIVE_ARTIFACTS_DIR)


def predict_price(area_m2, bedrooms, bathrooms, floors, age_years, distance_to_center_km, has_garage, has_garden, neighborhood_score):
    values = pd.DataFrame(
        [
            {
                "area_m2": area_m2,
                "bedrooms": bedrooms,
                "bathrooms": bathrooms,
                "floors": floors,
                "age_years": age_years,
                "distance_to_center_km": distance_to_center_km,
                "has_garage": has_garage,
                "has_garden": has_garden,
                "neighborhood_score": neighborhood_score,
            }
        ]
    )
    predicted = float(model.predict(values)[0])
    return f"Estimated house price: ${predicted:,.0f}"

interface = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Number(label="Area (m2)", value=180),
        gr.Number(label="Bedrooms", value=4),
        gr.Number(label="Bathrooms", value=3),
        gr.Number(label="Floors", value=2),
        gr.Number(label="Age (years)", value=12),
        gr.Number(label="Distance to center (km)", value=6.5),
        gr.Number(label="Has garage (0 or 1)", value=1),
        gr.Number(label="Has garden (0 or 1)", value=1),
        gr.Number(label="Neighborhood score", value=8.4),
    ],
    outputs=gr.Textbox(label="Prediction"),
    title="House Price Predictor",
    description="Colab-friendly demo interface for house price prediction.",
)

interface.launch(share=True, debug=False)